In [1]:
import re
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

import chess
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from utils.score_responses import ResponseEvaluator
from utils.parsing import parse_fen
from utils.board import get_piece_name_at_location

In [2]:
PARENT_DIR = Path("model_outputs")
DATA_FOLDERS = [
    "llmchess-qwen25-7b-400",
    "llmchess-llama4-maverick-400",
    "llmchess-oai-oss120b-low-400",
    "llmchess-qwen25-7b-sft-dm22-400",
    "eval-qwen25-7b-sftdm22-drgrpo-dm11-400",
    "llmchess-qwen25-7b-sft-dm23-400",
    "eval-qwen25-7b-sftdm23-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm24-400",
    "eval-qwen25-7b-sftdm24-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm25-400",
    "eval-qwen25-7b-sftdm25-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm26-400",
    "eval-qwen25-7b-sftdm26-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm27-400",
    "eval-qwen25-7b-sftdm27-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm28-400",
    "eval-qwen25-7b-sftdm28-drgrpo-dm10-400",
    "eval-qwen25-7b-sftdm28-drgrpo-dm10-take2-400",
    "llmchess-qwen25-7b-sft-dm29-p4-400",
    "eval-qwen25-7b-sftdm29-p4-drgrpo-dm12-400",
    "eval-qwen25-7b-sftdm29-p4-drgrpo-dm13-400",
    "llmchess-qwen25-7b-sft-dm30-400",
    "eval-qwen25-7b-sftdm30-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm31-400",
    "eval-qwen25-7b-sftdm31-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm32-400",
    "eval-qwen25-7b-sftdm32-drgrpo-dm12-400",
    "llmchess-qwen25-7b-sft-dm33-400",
    "eval-qwen25-7b-sftdm33-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm34-400",
    "eval-qwen25-7b-sftdm34-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm35-400",
    "eval-qwen25-7b-sftdm35-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm36-400",
    "eval-qwen25-7b-sftdm36-drgrpo-dm10-400",
    "llmchess-qwen25-7b-sft-dm29-p4-c37-400",
    "eval-qwen25-7b-sftdm29-p4-c37-drgrpo-dm12-400",
]

In [3]:
# ...existing code...

from pathlib import Path
import numpy as np
import pandas as pd

# Trivial move types (mirrors usage below)
TRIVIAL_MOVE_TYPES = {
    # "Standard Activate Knight": ["b1c3", "g1f3", "b8c6", "g8f6"],
    "Standard Activate Knight": [""],  # keep current placeholder to match notebook
    "Move Edge Pawn": ["a2a4", "a2a3", "h2h4", "h2h3", "a7a5", "a7a6", "h7h5", "h7h6"],
    "Trivial King/Rook": ["a1b1", "b1a1", "a8b8", "b8a8", "h1g1", "g1h1", "h8g8", "g8h8"],
}
TRIVIAL_MOVES_SET = {m.strip().lower() for moves in TRIVIAL_MOVE_TYPES.values() for m in moves if m}

def summarize_predictmove_for_folder(folder_path: Path, trivial_moves_set: set[str]):
    """
    Aggregates predictmove*.json/jsonl using only ResponseEvaluator outputs.
    Returns:
      top_answer_count (int),
      legal_move_rate (float),
      avg_legal_score (float),
      total_trivial_moves (int)  # counted over rows with numeric scores
    """
    total_generations = 0
    legal_generations = 0
    cumulative_score = 0.0
    top_answer_count = 0
    total_trivial_moves = 0

    for jf in folder_path.glob("*.json*"):
        if jf.suffix not in {".json", ".jsonl"} or "predictmove" not in jf.name:
            continue

        evaluator = ResponseEvaluator(str(folder_path), jf.name)
        res = evaluator.results
        total_generations += res.get("Count: Total Generations", 0)
        legal_generations += res.get("Count: Legal Generations", 0)
        cumulative_score += res.get("Total: Cumulative Score", 0.0)
        top_answer_count += res.get("Count: Top Answer", 0)

        # Count trivial moves from ResponseEvaluator's board_id_results (no extra recompute)
        for _, data in evaluator.board_id_results.items():
            for score_raw, move in data.get("score_answers", []):
                try:
                    s = float(score_raw)
                except (ValueError, TypeError):
                    s = np.nan
                if not np.isnan(s) and isinstance(move, str):
                    if move.strip().lower() in trivial_moves_set:
                        total_trivial_moves += 1

    legal_move_rate = (legal_generations / total_generations) if total_generations else np.nan
    avg_legal_score = (cumulative_score / legal_generations) if legal_generations else np.nan
    trivial_move_rate = (float(total_trivial_moves) / legal_generations) if legal_generations else np.nan
    return top_answer_count, legal_move_rate, avg_legal_score, trivial_move_rate


def summarize_legalmoves_for_folder(folder_path: Path):
    """
    Aggregates legalmoves*.json/jsonl using only ResponseEvaluator outputs.
    Returns:
      avg_legal_score (float)
    """
    total_legal, total_gens = 0, 0
    cumulative_score = 0.0

    for jf in folder_path.glob("*.json*"):
        if jf.suffix not in {".json", ".jsonl"} or "legalmoves" not in jf.name:
            continue
        evaluator = ResponseEvaluator(str(folder_path), jf.name)
        res = evaluator.results
        total_legal += res.get("Count: Legal Generations", 0)
        cumulative_score += res.get("Total: Cumulative Score", 0.0)
        total_gens += res.get("Count: Total Generations", 0)

    avg_legal = (cumulative_score / total_legal) if total_legal else np.nan
    legal_rate = (total_legal / total_gens) if total_legal else np.nan
    return avg_legal, legal_rate

def summarize_choose_from_n_for_folder(folder_path: Path, prefix: str):
    """
    Aggregates {prefix}*.json/jsonl using ResponseEvaluator outputs only.
    Returns:
      avg_legal_score (float)  # this is % accuracy for choose_from_n tasks
    """
    total_gens = 0
    cumulative_score = 0.0

    for jf in folder_path.glob("*.json*"):
        if jf.suffix not in {".json", ".jsonl"} or prefix not in jf.name:
            continue
        evaluator = ResponseEvaluator(str(folder_path), jf.name)
        res = evaluator.results
        total_gens += res.get("Count: Total Generations", 0)
        cumulative_score += res.get("Total: Cumulative Score", 0.0)

    return (cumulative_score / total_gens) if total_gens else np.nan



# Build summary rows for all experiments (DATA_FOLDERS)
rows = []
for data_folder in DATA_FOLDERS:
    folder_path = Path(PARENT_DIR) if data_folder is None else (Path(PARENT_DIR) / data_folder)
    if not folder_path.exists():
        print(f"Warning: folder not found: {folder_path}")
        rows.append({
            "experiment": str(data_folder),
            "predictmove - Top Answer": 0,
            "predictmove - legal move rate": np.nan,
            "predictmove - avg legal score": np.nan,
            "predictmove - trivial move rate": 0,
            "legalmoves - legal move rate": np.nan,
            "legalmoves - avg legal score": np.nan,
            "bestmove - avg score": np.nan,
            "worstmove - avg score": np.nan,
        })
        continue

    pm_top_cnt, pm_lmr, pm_avg, pm_trivial = summarize_predictmove_for_folder(folder_path, TRIVIAL_MOVES_SET)
    lm_avg, lm_lmr   = summarize_legalmoves_for_folder(folder_path)
    bm_avg   = summarize_choose_from_n_for_folder(folder_path, "bestmove")
    wm_avg   = summarize_choose_from_n_for_folder(folder_path, "worstmove")

    rows.append({
        "experiment": str(data_folder),
        "predictmove - Top Answer": int(pm_top_cnt),
        "predictmove - legal move rate": None if np.isnan(pm_lmr) else round(pm_lmr, 4),
        "predictmove - avg legal score": None if np.isnan(pm_avg) else round(pm_avg, 4),
        "predictmove - trivial move rate": pm_trivial,
        "legalmoves - legal move rate": None if np.isnan(lm_lmr) else round(lm_lmr, 4),
        "legalmoves - avg legal score": None if np.isnan(lm_avg) else round(lm_avg, 4),
        "bestmove - avg score": None if np.isnan(bm_avg) else round(bm_avg, 4),
        "worstmove - avg score": None if np.isnan(wm_avg) else round(wm_avg, 4),
    })

summary_df = pd.DataFrame(rows)
out_csv = Path(PARENT_DIR) / "summary_metrics.csv"
summary_df.to_csv(out_csv, index=False)
print(f"Wrote: {out_csv.resolve()} ({len(summary_df)} rows)")

--------------------------------------------------
Results for model_outputs\llmchess-qwen25-7b-400:
Filename: predictmove_eval_400_all_20250725-214147.json
Count: Sample Questions: 400
Count: Total Generations: 400
Count: Legal Generations: 32
Total: Cumulative Score: 17.992320038587078
Error: Illegal Move: 367
Error: Parsing: 1
Error: Other: 0
Count: Top Answer: 9
Avg. Score - All: 0.044980800096467696
Avg. Score - Legal: 0.5622600012058462
Legal Move Rate: 0.08
Error Rate: 0.92
--------------------------------------------------

--------------------------------------------------
Results for model_outputs\llmchess-qwen25-7b-400:
Filename: legalmoves_eval_400_all_20250725-214147.json
Count: Sample Questions: 400
Count: Total Generations: 400
Count: Legal Generations: 209
Total: Cumulative Score: 54.405827418195834
Error: Illegal Move: 0
Error: Parsing: 191
Error: Other: 0
Avg. Score - All: 0.13601456854548957
Avg. Score - Legal: 0.26031496372342505
Legal Move Rate: 0.5225
Error Rate: 

In [4]:
# Populate our board result dicts
board_result_dicts = {}
for data_folder in DATA_FOLDERS:
    folder_path = Path(PARENT_DIR) if data_folder is None else Path(PARENT_DIR) / data_folder
    for json_file in folder_path.glob("*.json*"):
        if json_file.suffix not in {".json", ".jsonl"} or "predictmove" not in json_file.name:
            continue
        key = f"{json_file.name} - [{data_folder}]"
        evaluator = ResponseEvaluator(str(folder_path), json_file.name)
        board_result_dicts[key] = evaluator.board_id_results

--------------------------------------------------
Results for model_outputs\llmchess-qwen25-7b-400:
Filename: predictmove_eval_400_all_20250725-214147.json
Count: Sample Questions: 400
Count: Total Generations: 400
Count: Legal Generations: 32
Total: Cumulative Score: 17.992320038587078
Error: Illegal Move: 367
Error: Parsing: 1
Error: Other: 0
Count: Top Answer: 9
Avg. Score - All: 0.044980800096467696
Avg. Score - Legal: 0.5622600012058462
Legal Move Rate: 0.08
Error Rate: 0.92
--------------------------------------------------

--------------------------------------------------
Results for model_outputs\llmchess-llama4-maverick-400:
Filename: predictmove_eval_400_all_20250712-230204.json
Count: Sample Questions: 400
Count: Total Generations: 400
Count: Legal Generations: 166
Total: Cumulative Score: 102.54727933108144
Error: Illegal Move: 234
Error: Parsing: 0
Error: Other: 0
Count: Top Answer: 54
Avg. Score - All: 0.2563681983277036
Avg. Score - Legal: 0.6177546947655509
Legal Mov

In [5]:
def bar_by(df, x_col, y_col, *, n_bins=10, title=None):
    """
    Dual‑axis bar plot:
      • left  y‑axis: mean(y_col) per bucket (blue)
      • right y‑axis: count per bucket (orange)

    Buckets: unique values if ≤10 else `n_bins` equal‑width numeric buckets.
    """
    x, y = df[x_col], df[y_col]

    # ── Bucket assignment ────────────────────────────────────────────
    if pd.api.types.is_numeric_dtype(x):
        if x.nunique() <= 10:
            labels = sorted(x.unique())
            grp_idx = x
        else:
            bins   = np.linspace(x.min(), x.max(), n_bins + 1)
            labels = [f"{int(bins[i])}–{int(bins[i+1]-1)}" for i in range(n_bins)]
            grp_idx = pd.cut(x, bins=bins, labels=labels, include_lowest=True)
    else:  # categorical x
        labels  = sorted(x.astype(str).unique())
        grp_idx = x.astype(str)

    # ── Aggregation ──────────────────────────────────────────────────
    mean_vals  = y.groupby(grp_idx, observed=True).mean().reindex(labels)
    count_vals = y.groupby(grp_idx, observed=True).count().reindex(labels).fillna(0)

    # ── Plot ─────────────────────────────────────────────────────────
    xpos, width = np.arange(len(labels)), 0.35
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    bars_mean  = ax1.bar(xpos - width/2, mean_vals.values, width, color="#1f77b4")
    bars_count = ax2.bar(xpos + width/2, count_vals.values, width,
                         color="orange", label="# samples")

    ax1.set_xlabel(x_col)
    ax1.set_ylabel(f"mean({y_col})")
    ax2.set_ylabel("count")
    ax1.set_xticks(xpos)
    ax1.set_xticklabels(labels, rotation=45, ha="right")
    ax1.set_title(title or f"{y_col} by {x_col}")

    # Annotate bars
    for rect in bars_mean:
        ax1.text(rect.get_x() + rect.get_width()/2,
                 rect.get_height() + 0.01,
                 f"{rect.get_height():.2f}",
                 ha="center", va="bottom", fontsize=8)
    for rect in bars_count:
        ax2.text(rect.get_x() + rect.get_width()/2,
                 rect.get_height() + 0.01,
                 f"{int(rect.get_height())}",
                 ha="center", va="bottom", fontsize=8)

    ax1.legend([bars_count], ["# samples"], loc="upper right")
    fig.tight_layout()
    plt.show()

# Predict Move   
---

In [6]:
plot_move_stats = True

# Trivial Move Types
move_type = {
    # "Standard Activate Knight": ["b1c3", "g1f3", "b8c6", "g8f6"],
    "Standard Activate Knight": [""],
    "Move Edge Pawn": ["a2a4", "a2a3", "h2h4", "h2h3", "a7a5", "a7a6", "h7h5", "h7h6"],
    "Trivial King/Rook": ["a1b1", "b1a1", "a8b8", "b8a8", "h1g1", "g1h1", "h8g8", "g8h8"]
}

if plot_move_stats:
    for key, results in board_result_dicts.items():
        if not key.startswith("predictmove"):
            continue

        rows = []
        for board_id, data in results.items():
            fen = data['info']["board"]
            parsed = parse_fen(fen)
            fullmove = parsed["fullmove_number"]

            # Build a python-chess board with proper turn side
            try:
                board = chess.Board(fen)
            except Exception:
                board = None

            for score_raw, move in data.get("score_answers", []):
                try:         # numeric score or NaN for '<ERROR>'
                    score = float(score_raw)
                except (ValueError, TypeError):
                    score = np.nan

                piece_name = "na"  # aggregate by piece only, not color
                if board is not None and isinstance(move, str):
                    # Moves are expected as UCI (e.g., 'e2e4', 'e7e8q').
                    try:
                        uci = move.strip().lower()
                        m = chess.Move.from_uci(uci)
                        if m in board.legal_moves:
                            p = board.piece_at(m.from_square)
                            if p is not None:
                                piece_map = {
                                    chess.PAWN: "pawn",
                                    chess.KNIGHT: "knight",
                                    chess.BISHOP: "bishop",
                                    chess.ROOK: "rook",
                                    chess.QUEEN: "queen",
                                    chess.KING: "king",
                                }
                                piece_name = piece_map.get(p.piece_type, "unknown")
                        # Do not push the move; we analyze per-row and keep board at FEN state
                    except Exception:
                        piece_name = "na"

                rows.append({
                    "board_id":   board_id,
                    "fullmove":   fullmove,
                    "move":       move,
                    "score":      score,
                    "piece_type": piece_name
                })

        if not rows:
            continue

        df = pd.DataFrame(rows)

        # ── 1) Histogram (mean score vs fullmove buckets) ────────────
        clean_df = df.dropna(subset=["score"])
        # if not clean_df.empty:
        #     df["fullmove_bucket"] = (df["fullmove"] // 10) * 10
        #     bar_by(
        #         df.dropna(subset=["score"]),
        #         x_col="fullmove_bucket",
        #         y_col="score",
        #         title=f"{key}: Avg. Score vs Fullmove Count"
        #     )

        # ── 2) Move statistics ───────────────────────────────────────
        print(f"\n{key} — Top 10 moves [total legal: {len(clean_df)}]:")
        move_counts = clean_df["move"].value_counts().head(10)
        move_stats  = (
            clean_df.groupby("move")["score"]
                .agg(["mean", "std"])
                .reindex(move_counts.index)
        )
        for mv in move_counts.index:
            cnt  = move_counts[mv]
            mean = move_stats.loc[mv, "mean"]
            std  = move_stats.loc[mv, "std"]
            mean_str = f"{mean:.3f}" if not np.isnan(mean) else "—"
            std_str  = f"{std:.3f}"  if not np.isnan(std)  else "—"
            print(f"{mv:<8} count={cnt:>5} mean={mean_str:>7} std={std_str:>7}")

        # ── 3) Trivial Move Type Counts ──────────────────────────────
        total_moves = len(clean_df)
        total_trivial_moves = 0
        print(f"\nTrivial Move Type Counts:")
        for mt_key, mt_moves in move_type.items():
            mt_count = clean_df["move"].isin(mt_moves).sum()
            percent = (mt_count / total_moves * 100) if total_moves > 0 else 0.0
            total_trivial_moves += mt_count
            print(f"   {mt_key:<25}: {mt_count:>5} [{percent:>5.0f}%]")
        
        print(f"{'Total Trivial Moves':<28}: {total_trivial_moves:>5} [{(total_trivial_moves/total_moves)*100:>5.0f}%]")
        
        
        # -- 4) Count and score by piece type --------------------------
        # Print out the different piece types and the number associated
        # as well as the mean score (to 3 decimals)
        piece_df = clean_df.copy()
        piece_df["piece_type"] = piece_df["piece_type"].fillna("na")
        piece_stats = piece_df.groupby("piece_type")["score"].agg(["count", "mean"]).sort_values("count", ascending=False)
        print("\nPiece Type Counts and Mean Scores:")
        for idx, row in piece_stats.iterrows():
            mean_val = row["mean"]
            mean_str = f"{mean_val:.3f}" if not np.isnan(mean_val) else "—"
            print(f"   {idx:<15} count={int(row['count']):>6} mean={mean_str:>7}")
        print("-" * 60)


predictmove_eval_400_all_20250725-214147.json - [llmchess-qwen25-7b-400] — Top 10 moves [total legal: 32]:
g7g5     count=    4 mean=  0.548 std=  0.279
e1g1     count=    3 mean=  0.822 std=  0.153
f6e4     count=    3 mean=  0.347 std=  0.172
e7e5     count=    2 mean=  0.868 std=  0.123
f2f4     count=    2 mean=  0.088 std=  0.001
g2g3     count=    1 mean=  0.405 std=      —
f6g4     count=    1 mean=  0.750 std=      —
f7f6     count=    1 mean=  0.086 std=      —
a1a3     count=    1 mean=  0.714 std=      —
h7h6     count=    1 mean=  0.844 std=      —

Trivial Move Type Counts:
   Standard Activate Knight :     0 [    0%]
   Move Edge Pawn           :     2 [    6%]
   Trivial King/Rook        :     0 [    0%]
Total Trivial Moves         :     2 [    6%]

Piece Type Counts and Mean Scores:
   pawn            count=    17 mean=  0.547
   knight          count=     7 mean=  0.425
   king            count=     4 mean=  0.843
   rook            count=     3 mean=  0.676
   bishop

In [11]:
# Set this to True to show smoothed line (KDE) plots, or False for bar charts
show_kde = False  # <-- Change to True for smoothed line charts

def get_short_name(folder):
    name = re.sub(r"^llmchess-qwen25-7b-", "", folder)
    name = re.sub(r"^eval-qwen25-7b-", "", name)
    name = re.sub(r"-400$", "", name)
    return name

def get_scores_for_folder(folder):
    scores = []
    for key, results in board_result_dicts.items():
        if folder in key and key.startswith("predictmove"):
            for board_id, data in results.items():
                for score_raw, _ in data.get("score_answers", []):
                    try:
                        score = float(score_raw)
                    except (ValueError, TypeError):
                        score = np.nan
                    if not np.isnan(score):
                        scores.append(score)
    return scores

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Expect NAME_MAPPING to already exist in your env:
# NAME_MAPPING = {"raw_folder_key": "Nice Label", ...}

SFT_COLOR = "#FECE9E"   # tan
RL_COLOR  = "#7A945A"   # dark green
SFT_DARK  = "#E0B885"   # slightly darker tan for text
RL_DARK   = "#5F7748"   # slightly darker green for text
DEFAULT_COLOR = "#9aa0a6"
DEFAULT_DARK  = "#6b7380"


NAME_MAPPING = {
    "llmchess-qwen25-7b-sft-dm29-p4-400": "Final SFT",
    "eval-qwen25-7b-sftdm29-p4-drgrpo-dm12-400": "Final RL"
}

def pretty_hist(series_dict, bin_edges=np.linspace(0, 1, 11),
                xlabel="Move Rank",
                ylabel="Legal Move Count",
                title="Legal Move Distribution (Out of 400)",
                fig_size=(8.8, 5.2)):
    """
    series_dict: {key -> 1D array-like of scores in [0,1]}
    NAME_MAPPING maps a filename (full or short) -> pretty label that contains "(SFT)" or "(RL)".
    """

    def _nice_label(key: str) -> str:
        if key in NAME_MAPPING:
            return NAME_MAPPING[key]
        try:
            short = get_short_name(key)
            if short in NAME_MAPPING:
                return NAME_MAPPING[short]
        except Exception:
            short = key
        for k, v in NAME_MAPPING.items():
            try:
                if k.endswith(key) or get_short_name(k) == key:
                    return v
            except Exception:
                if k.endswith(key):
                    return v
        return key

    sns.set_theme(context="talk", style="white")  # no background grid
    fig, ax = plt.subplots(figsize=fig_size)

    for key, scores in series_dict.items():
        scores = np.asarray(scores, dtype=float)
        scores = scores[~np.isnan(scores)]
        if scores.size == 0:
            continue

        nice = _nice_label(key)
        label = f"{nice} (n={scores.size})"

        lower_nice = nice.lower()
        if "(sft)" in lower_nice:
            color, dark = SFT_COLOR, SFT_DARK
        elif "(rl)" in lower_nice:
            color, dark = RL_COLOR, RL_DARK
        else:
            key_l = str(key).lower()
            if "sft" in key_l:
                color, dark = SFT_COLOR, SFT_DARK
            elif "rl" in key_l:
                color, dark = RL_COLOR, RL_DARK
            else:
                color, dark = DEFAULT_COLOR, DEFAULT_DARK

        sns.histplot(
            scores,
            bins=bin_edges,
            stat="count",
            element="bars",
            fill=True,
            alpha=0.5,
            edgecolor="white",
            linewidth=0.6,
            shrink=0.97,
            ax=ax,
            label=label,
            color=color,
        )

        med = float(np.median(scores))
        ax.axvline(med, linestyle="--", linewidth=2.0, alpha=0.9, color=color)

        # median text: just left of the line, horizontal, darker, larger, alpha=1
        xmin, xmax = ax.get_xlim()
        xspan = xmax - xmin if xmax > xmin else 1.0
        x_text = med - 0.005 * xspan  # closer to the line
        y_text = ax.get_ylim()[1] * 0.96
        ax.text(
            x_text, y_text, f"median={med:.2f}",
            va="top", ha="right", fontsize=12, rotation=0,
            color=dark, alpha=1.0
        )

    # axes labels/title
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=12)

    # only horizontal solid light gray lines
    ax.grid(False)
    ax.yaxis.grid(True, which="major", linestyle="-", linewidth=0.8, color="#d0d0d0")
    ax.xaxis.grid(False)
    ax.minorticks_off()

    sns.despine(ax=ax, left=False, bottom=False)

    # reverse legend order only (keep plotting order unchanged)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[::-1], labels[::-1], frameon=False, loc="upper left")

    fig.tight_layout()
    return fig, ax



FOLDER_COMBINATIONS = [
    # ["eval-qwen25-7b-sftdm22-drgrpo-dm11-400", "llmchess-qwen25-7b-sft-dm22-400"],
    # ["eval-qwen25-7b-sftdm23-drgrpo-dm10-400", "llmchess-qwen25-7b-sft-dm23-400"],
    # ["eval-qwen25-7b-sftdm24-drgrpo-dm10-400", "llmchess-qwen25-7b-sft-dm24-400"],
    # ["eval-qwen25-7b-sftdm25-drgrpo-dm10-400", "llmchess-qwen25-7b-sft-dm25-400"],
    # ["eval-qwen25-7b-sftdm26-drgrpo-dm10-400", "llmchess-qwen25-7b-sft-dm26-400"],
    # ["eval-qwen25-7b-sftdm27-drgrpo-dm10-400", "llmchess-qwen25-7b-sft-dm27-400"],
    # ["eval-qwen25-7b-sftdm28-drgrpo-dm10-400", "eval-qwen25-7b-sftdm28-drgrpo-dm10-take2-400", "llmchess-qwen25-7b-sft-dm28-400"],
    # ["eval-qwen25-7b-sftdm29-p4-drgrpo-dm12-400", "eval-qwen25-7b-sftdm29-p4-drgrpo-dm13-400", "llmchess-qwen25-7b-sft-dm29-p4-400"],
    # ["eval-qwen25-7b-sftdm29-p4-drgrpo-dm12-400", "llmchess-qwen25-7b-sft-dm29-p4-400"],
    # ["eval-qwen25-7b-sftdm29-p1-drgrpo-dm13-400", "llmchess-qwen25-7b-sft-dm29-p1-400"],
    # ["eval-qwen25-7b-sftdm30-drgrpo-dm10-400", "llmchess-qwen25-7b-sft-dm30-400"],
    # ["eval-qwen25-7b-sftdm31-drgrpo-dm10-400", "llmchess-qwen25-7b-sft-dm31-400"],
    # ["eval-qwen25-7b-sftdm32-drgrpo-dm12-400", "llmchess-qwen25-7b-sft-dm32-400"],
    # ["eval-qwen25-7b-sftdm32-drgrpo-dm12-400", "llmchess-qwen25-7b-sft-dm32-400"],
    ["llmchess-qwen25-7b-sft-dm29-p4-c37-400", "eval-qwen25-7b-sftdm29-p4-c37-drgrpo-dm12-400"],
]


bin_edges = np.arange(0, 1.0001, 0.1)

# for combo in FOLDER_COMBINATIONS:
#     series = {}
#     for folder in combo:
#         scores = get_scores_for_folder(folder)
#         if scores:
#             series[get_short_name(folder)] = scores
#     if series:
#         pretty_hist(
#             series_dict=series,
#             bin_edges=bin_edges,
#         )
#         plt.show()

for combo in FOLDER_COMBINATIONS:
    for folder in combo:
        scores = get_scores_for_folder(folder)
        if not scores:
            continue
        scores = np.asarray(scores, dtype=float)
        scores = scores[~np.isnan(scores)]
        counts, _ = np.histogram(scores, bins=bin_edges)
        print(f"{get_short_name(folder)}:")
        for i in range(len(bin_edges)-1):
            print(f"  {bin_edges[i]:.1f}–{bin_edges[i+1]:.1f}: {counts[i]}")
        print(f"  median={np.median(scores):.2f}")
        print(f"  mean={np.mean(scores):.2f}\n")


sft-dm29-p4-c37:
  0.0–0.1: 29
  0.1–0.2: 16
  0.2–0.3: 14
  0.3–0.4: 18
  0.4–0.5: 13
  0.5–0.6: 11
  0.6–0.7: 16
  0.7–0.8: 22
  0.8–0.9: 18
  0.9–1.0: 58
  median=0.66
  mean=0.58

sftdm29-p4-c37-drgrpo-dm12:
  0.0–0.1: 8
  0.1–0.2: 5
  0.2–0.3: 8
  0.3–0.4: 9
  0.4–0.5: 15
  0.5–0.6: 16
  0.6–0.7: 18
  0.7–0.8: 28
  0.8–0.9: 63
  0.9–1.0: 222
  median=0.93
  mean=0.83



# Legal Moves   
---

In [8]:
for key, results in board_result_dicts.items():
    if not key.startswith("legalmoves"):
        continue

    # Compute average IoU for each

    # Get mean score by piece (w/ count overlay)
    # rows = []
    # for board_id, data in results.items():
    #     piece = data["info"]["task_data"][0]   # e.g. 'black bishop'
    #     for score_raw, _ in data.get("score_answers", []):
    #         try:
    #             score = float(score_raw)      # 0‥1 numeric
    #         except (ValueError, TypeError):
    #             score = pd.NA                 # '<ERROR>' → NA
    #         rows.append({"piece": piece, "score": score})

    # df = pd.DataFrame(rows).dropna(subset=["score"]).astype({"score": float})
    # if df.empty:
    #     print(f"{key}: no valid scores")
    #     continue

    # bar_by(
    #     df,
    #     x_col="piece",
    #     y_col="score",
    #     title=f"{key}: mean score by piece (count overlay)"
    # )

# Best Move   
---

In [9]:
for key, results in board_result_dicts.items():
    if not key.startswith("bestmove"):
        continue

    rows = []
    for board_id, data in results.items():
        fen       = data["info"]["board"]
        fullmove  = parse_fen(fen)["fullmove_number"]
        # correct answer square = first 2 chars of SAN‑less move string
        src_sq    = data["info"]["answer"]["answer"][:2].lower()

        piece = get_piece_name_at_location(fen, src_sq)  # e.g. 'white knight'

        for score_raw, _ in data.get("score_answers", []):
            try:
                score = float(score_raw)
            except (ValueError, TypeError):
                score = np.nan
            rows.append({
                "board_id": board_id,
                "fullmove": fullmove,
                "piece":    piece,
                "score":    score,
            })

    df = pd.DataFrame(rows).dropna(subset=["score"]).astype({"score": float})
    if df.empty:
        print(f"{key}: no valid scores.")
        continue

    # ─── 1) Score vs full‑move count buckets ────────────────────
    df["fullmove_bucket"] = (df["fullmove"] // 10) * 10
    # bar_by(
    #     df.dropna(subset=["score"]),
    #     x_col="fullmove_bucket",
    #     y_col="score",
    #     title=f"{key}: Avg. Score vs Fullmove Count"
    # )

    # ─── 2) Score vs piece type ─────────────────────────────────
    bar_by(
        df,
        x_col="piece",
        y_col="score",
        title=f"{key}: mean score by piece (count overlay)"
    )

# Worst Move   
---

In [10]:
for key, results in board_result_dicts.items():
    if not key.startswith("worstmove"):
        continue

    rows = []
    for board_id, data in results.items():
        fen       = data["info"]["board"]
        fullmove  = parse_fen(fen)["fullmove_number"]
        # correct answer square = first 2 chars of SAN‑less move string
        src_sq    = data["info"]["answer"]["answer"][:2].lower()

        piece = get_piece_name_at_location(fen, src_sq)  # e.g. 'white knight'

        for score_raw, _ in data.get("score_answers", []):
            try:
                score = float(score_raw)
            except (ValueError, TypeError):
                score = np.nan
            rows.append({
                "board_id": board_id,
                "fullmove": fullmove,
                "piece":    piece,
                "score":    score,
            })

    df = pd.DataFrame(rows).dropna(subset=["score"]).astype({"score": float})
    if df.empty:
        print(f"{key}: no valid scores.")
        continue

    # ─── 1) Score vs full‑move count buckets ────────────────────
    df["fullmove_bucket"] = (df["fullmove"] // 10) * 10
    bar_by(
        df.dropna(subset=["score"]),
        x_col="fullmove_bucket",
        y_col="score",
        title=f"{key}: Avg. Score vs Fullmove Count"
    )

    # ─── 2) Score vs piece type ─────────────────────────────────
    bar_by(
        df,
        x_col="piece",
        y_col="score",
        title=f"{key}: mean score by piece (count overlay)"
    )